In [3]:
import pandas as pd
import numpy as np
import sklearn
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Scikit-learn version:", sklearn.__version__)

# قراءة الداتا
df = pd.read_csv('/diabetes.csv')
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Scikit-learn version: 1.6.1
Dataset Shape: (768, 9)
Columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
# اختيار أهم 4 مدخلات يقدر أي مستخدم يكتبها بسهولة في الشات
FEATURES = ['Glucose', 'BloodPressure', 'BMI', 'Age']
TARGET = 'Outcome'

X = df[FEATURES].copy()
y = df[TARGET].copy()

# التأكد من عدد الأصفار غير المنطقية في القياسات الطبية
cols_with_invalid_zeros = ['Glucose', 'BloodPressure', 'BMI']
print("Zeros BEFORE cleaning:")
print((X[cols_with_invalid_zeros] == 0).sum())

# تحويل الأصفار لقيم مفقودة (NaN) عشان الـ Imputer يعالجها صح
X[cols_with_invalid_zeros] = X[cols_with_invalid_zeros].replace(0, np.nan)

print("\nMissing values (NaN) AFTER replacing zeros:")
print(X.isnull().sum())

Zeros BEFORE cleaning:
Glucose           5
BloodPressure    35
BMI              11
dtype: int64

Missing values (NaN) AFTER replacing zeros:
Glucose           5
BloodPressure    35
BMI              11
Age               0
dtype: int64


In [5]:
# تقسيم الداتا 80% تدريب و 20% اختبار مع الحفاظ على نسبة الفئات
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# بناء Pipeline يجمع التنظيف والـ Scaling والموديل في كيان واحد
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])

# تدريب الـ Pipeline بالكامل على بيانات التدريب فقط
pipeline.fit(X_train, y_train)
print("Model training completed successfully!")

Model training completed successfully!


In [6]:
y_pred = pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Diabetes (0)', 'Diabetes Risk (1)']))

Accuracy: 70.78%

Confusion Matrix:
[[81 19]
 [26 28]]

Classification Report:
                   precision    recall  f1-score   support

  No Diabetes (0)       0.76      0.81      0.78       100
Diabetes Risk (1)       0.60      0.52      0.55        54

         accuracy                           0.71       154
        macro avg       0.68      0.66      0.67       154
     weighted avg       0.70      0.71      0.70       154



In [7]:
model_filename = 'diabetes_model.joblib'
joblib.dump(pipeline, model_filename)
print(f"Model saved successfully to {model_filename}")

Model saved successfully to diabetes_model.joblib


In [8]:
# تحميل الموديل من الملف المحفوظ وتجربته على حالتين
loaded_model = joblib.load('diabetes_model.joblib')

test_cases = pd.DataFrame([
    {'Glucose': 95,  'BloodPressure': 70, 'BMI': 22.5, 'Age': 24},  # حالة طبيعية
    {'Glucose': 175, 'BloodPressure': 88, 'BMI': 35.2, 'Age': 52}   # حالة خطر عالي
])

probs = loaded_model.predict_proba(test_cases)[:, 1]

for i, prob in enumerate(probs):
    print(f"Case {i+1} -> Diabetes Risk Probability: {prob:.2%}")

Case 1 -> Diabetes Risk Probability: 4.05%
Case 2 -> Diabetes Risk Probability: 87.55%


In [9]:
%%writefile ml_tool.py
import os
import json
import joblib
import pandas as pd

# تحديد مسار الموديل نسبة لمكان الملف ده عشان ميحصلش FileNotFoundError عند باقي التيم
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH = os.path.join(BASE_DIR, "diabetes_model.joblib")

# تحميل الموديل مرة واحدة بس وقت استدعاء الملف عشان السرعة
try:
    _model = joblib.load(MODEL_PATH)
except Exception as e:
    _model = None
    _load_error = str(e)
else:
    _load_error = None


def _categorize_risk(probability: float) -> str:
    """تقسيم نسبة الاحتمالية لـ 3 فئات خطر واضحة."""
    if probability < 0.35:
        return "Low Risk"
    elif probability < 0.65:
        return "Moderate Risk"
    else:
        return "High Risk"


def predict_diabetes_risk(glucose: float, blood_pressure: float, bmi: float, age: float) -> str:
    """
    Predicts diabetes-related risk category using basic health measurements.

    Args:
        glucose (float): Plasma glucose concentration in mg/dL (valid range: 40 to 400).
        blood_pressure (float): Diastolic blood pressure in mm Hg (valid range: 30 to 200).
        bmi (float): Body Mass Index in kg/m^2 (valid range: 10 to 80).
        age (float): Age in years (valid range: 1 to 120).

    Returns:
        str: A JSON-formatted string containing risk category, probability,
             inputs used, and an educational disclaimer (or an error message).
    """
    # 1. التأكد إن الموديل اتحمل بنجاح
    if _model is None:
        return json.dumps({
            "status": "error",
            "error_type": "model_load_error",
            "message": f"Failed to load ML model: {_load_error}"
        }, indent=2)

    # 2. التحقق من إن المدخلات أرقام فعلاً (Input Type Validation)
    raw_inputs = {
        "Glucose": glucose,
        "BloodPressure": blood_pressure,
        "BMI": bmi,
        "Age": age
    }

    parsed_inputs = {}
    for key, value in raw_inputs.items():
        if value is None or isinstance(value, bool):
            return json.dumps({
                "status": "error",
                "error_type": "missing_or_invalid_input",
                "message": f"Invalid or missing value for '{key}'. Please provide a valid number."
            }, indent=2)
        try:
            parsed_inputs[key] = float(value)
        except (ValueError, TypeError):
            return json.dumps({
                "status": "error",
                "error_type": "invalid_type",
                "message": f"'{key}' must be a numeric value, got '{value}'."
            }, indent=2)

    # 3. التحقق من المنطقية الطبية للأرقام (Range Validation)
    valid_ranges = {
        "Glucose": (40, 400),
        "BloodPressure": (30, 200),
        "BMI": (10, 80),
        "Age": (1, 120)
    }

    for key, (min_val, max_val) in valid_ranges.items():
        val = parsed_inputs[key]
        if not (min_val <= val <= max_val):
            return json.dumps({
                "status": "error",
                "error_type": "out_of_range",
                "message": f"'{key}' value ({val}) is out of realistic range ({min_val} to {max_val})."
            }, indent=2)

    # 4. تنفيذ التوقع وترجيع النتيجة كـ JSON منظم
    try:
        input_df = pd.DataFrame([parsed_inputs])
        prob = float(_model.predict_proba(input_df)[0][1])
        risk_category = _categorize_risk(prob)

        result = {
            "status": "success",
            "risk_category": risk_category,
            "risk_probability_percent": round(prob * 100, 2),
            "inputs_used": parsed_inputs,
            "disclaimer": "This is an educational risk estimate, not a medical diagnosis or treatment recommendation."
        }
        return json.dumps(result, indent=2)

    except Exception as e:
        return json.dumps({
            "status": "error",
            "error_type": "prediction_error",
            "message": f"An unexpected error occurred during prediction: {str(e)}"
        }, indent=2)

Writing ml_tool.py


In [10]:
from ml_tool import predict_diabetes_risk

print("=== Test 1: Low Risk Case ===")
print(predict_diabetes_risk(glucose=95, blood_pressure=70, bmi=22.5, age=24))

print("\n=== Test 2: Moderate Risk Case ===")
print(predict_diabetes_risk(glucose=135, blood_pressure=80, bmi=31.0, age=40))

print("\n=== Test 3: High Risk Case ===")
print(predict_diabetes_risk(glucose=175, blood_pressure=88, bmi=35.2, age=52))

print("\n=== Test 4: Invalid Input (Negative Age) ===")
print(predict_diabetes_risk(glucose=120, blood_pressure=75, bmi=25.0, age=-5))

print("\n=== Test 5: Invalid Type (Text instead of Number) ===")
print(predict_diabetes_risk(glucose="high", blood_pressure=75, bmi=25.0, age=30))

=== Test 1: Low Risk Case ===
{
  "status": "success",
  "risk_category": "Low Risk",
  "risk_probability_percent": 4.05,
  "inputs_used": {
    "Glucose": 95.0,
    "BloodPressure": 70.0,
    "BMI": 22.5,
    "Age": 24.0
  },
  "disclaimer": "This is an educational risk estimate, not a medical diagnosis or treatment recommendation."
}

=== Test 2: Moderate Risk Case ===
{
  "status": "success",
  "risk_category": "Moderate Risk",
  "risk_probability_percent": 41.75,
  "inputs_used": {
    "Glucose": 135.0,
    "BloodPressure": 80.0,
    "BMI": 31.0,
    "Age": 40.0
  },
  "disclaimer": "This is an educational risk estimate, not a medical diagnosis or treatment recommendation."
}

=== Test 3: High Risk Case ===
{
  "status": "success",
  "risk_category": "High Risk",
  "risk_probability_percent": 87.55,
  "inputs_used": {
    "Glucose": 175.0,
    "BloodPressure": 88.0,
    "BMI": 35.2,
    "Age": 52.0
  },
  "disclaimer": "This is an educational risk estimate, not a medical diagnosis 

In [11]:
from google.colab import files
files.download('diabetes_model.joblib')
files.download('ml_tool.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>